# ReID Pipeline – Step 1: Detection + Tracking (BoxMOT)
**Kaggle dual T4 – RT‑DETRv2‑L + BoT‑SORT**

1. Install dependencies
2. Download model
3. Build module & config
4. Process video → crops per local track + annotated video

In [ ]:
# ---------- 1. Install dependencies ----------
!pip install -U ultralytics boxmot opencv-python-headless numpy tqdm

In [ ]:
# ---------- 2. Download RT-DETRv2-L model ----------
# Ultralytics provides RT-DETR weights – we use 'rtdetr-l.pt'
from ultralytics import YOLO
import torch

# This will download the model from Ultralytics' server
model = YOLO('rtdetr-l.pt')   # automatically downloads if not present
print('Model loaded on', next(model.model.parameters()).device)

In [ ]:
# ---------- 3. Set up file structure ----------
import os
from pathlib import Path

# Kaggle working directory
BASE_DIR = Path('/kaggle/working') / 'reid_system'
BASE_DIR.mkdir(parents=True, exist_ok=True)
MODULES_DIR = BASE_DIR / 'modules'
MODULES_DIR.mkdir(exist_ok=True)
OUTPUT_DIR = BASE_DIR / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

# Path to your uploaded video (if you uploaded it as a dataset, adjust)
# For this notebook, assume video is at /kaggle/input/yourvideo/store_cam3.mp4
# If you uplaod the video directly, it will be in /kaggle/input.
VIDEO_INPUT = '/kaggle/input/your-dataset-name/store_cam3.mp4'

# Write config.py
config_content = """
from pathlib import Path

BASE_DIR = Path('/kaggle/working/reid_system')
MODELS_DIR = BASE_DIR / 'models'
OUTPUT_DIR = BASE_DIR / 'outputs'

INPUT_VIDEO = '/kaggle/input/your-dataset-name/store_cam3.mp4'

DEVICE_YOLO = 'cuda:0'   # first T4
YOLO_MODEL = 'rtdetr-l.pt'
TRACKER_CONFIG = 'botsort.yaml'   # BoxMOT built-in
TRACKER_MODEL = 'osnet_x1_0.onnx'  # placeholder (will be replaced later)
DET_CONFIDENCE = 0.25

OUTPUT_VIDEO = str(OUTPUT_DIR / 'tracked_output.mp4')
"""
with open(BASE_DIR / 'config.py', 'w') as f:
    f.write(config_content.strip())

print('Project structure ready.')
print('Config:')
print(config_content)

In [ ]:
# ---------- 4. Create detector_tracker.py module ----------
module_code = """
"""Detector + Tracker – BoxMOT (RT-DETRv2 + BoT-SORT)"""
from typing import List, Dict, Tuple, Optional
import numpy as np
import cv2
from boxmot import BoTSORT
from ultralytics import YOLO
import torch
import config

class DetectorTracker:
    def __init__(self, model_path=config.YOLO_MODEL,
                 device=config.DEVICE_YOLO, conf=config.DET_CONFIDENCE):
        self.detector = YOLO(model_path)
        self.detector.to(device)
        self.tracker = BoTSORT(
            track_high_thresh=conf,
            track_low_thresh=0.1,
            new_track_thresh=conf,
            track_buffer=30,
            match_thresh=0.8,
        )
        self.device = device
        self.conf = conf

    def track(self, frame: np.ndarray) -> List[Dict]:
        # Run detector
        results = self.detector.track(frame, persist=True, tracker='',
                                      classes=[0], conf=self.conf,
                                      device=self.device, imgsz=640,
                                      verbose=False)
        dets = []
        if results[0].boxes is not None and len(results[0].boxes):
            boxes = results[0].boxes.xyxy.cpu().numpy()
            confs = results[0].boxes.conf.cpu().numpy()
            for box, conf in zip(boxes, confs):
                x1, y1, x2, y2 = box
                dets.append([x1, y1, x2, y2, conf, 0])   # class=0 person
        dets = np.array(dets) if dets else np.empty((0, 6))

        # Update tracker
        tracked = self.tracker.update(dets, frame)

        output = []
        for t in tracked:
            x1, y1, x2, y2, local_id = int(t[0]), int(t[1]), int(t[2]), int(t[3]), int(t[4])
            conf = float(t[6]) if len(t) > 6 else self.conf
            crop = self._safe_crop(frame, x1, y1, x2, y2)
            if crop is not None:
                output.append({'local_id': local_id, 'bbox': (x1,y1,x2,y2),
                               'crop': crop, 'conf': conf})
        return output

    def _safe_crop(self, frame, x1, y1, x2, y2):
        h, w = frame.shape[:2]
        x1c, y1c = max(0, min(x1, w-1)), max(0, min(y1, h-1))
        x2c, y2c = max(0, min(x2, w)), max(0, min(y2, h))
        if x2c <= x1c or y2c <= y1c:
            return None
        return frame[y1c:y2c, x1c:x2c]
"""
with open(MODULES_DIR / 'detector_tracker.py', 'w') as f:
    f.write(module_code.strip())

print('detector_tracker.py written.')

In [ ]:
# ---------- 5. Run the experiment ----------
import sys, time
from pathlib import Path
import cv2
import numpy as np
from tqdm import tqdm

# Add project root to path
sys.path.insert(0, str(BASE_DIR))
import config
from modules.detector_tracker import DetectorTracker

def id_to_color(track_id):
    hue = ((track_id * 0.61803398875) % 1.0) * 179.0
    hsv = np.array([[[hue, 220, 255]]], dtype=np.uint8)
    bgr = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)[0, 0]
    return int(bgr[0]), int(bgr[1]), int(bgr[2])

crops_dir = config.OUTPUT_DIR / 'step1_crops'
crops_dir.mkdir(exist_ok=True)

cap = cv2.VideoCapture(config.INPUT_VIDEO)
assert cap.isOpened(), f"Cannot open {config.INPUT_VIDEO}"
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width, height = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

out = cv2.VideoWriter(config.OUTPUT_VIDEO, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

tracker = DetectorTracker()
frame_idx = 0
saved_total = 0
saved_per_track = {}

pbar = tqdm(total=total_frames, desc='Tracking')
start = time.perf_counter()

while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame_idx += 1
    detections = tracker.track(frame)
    annotated = frame.copy()
    for det in detections:
        lid = det['local_id']
        x1,y1,x2,y2 = det['bbox']
        color = id_to_color(lid)
        cv2.rectangle(annotated, (x1,y1), (x2,y2), color, 2)
        cv2.putText(annotated, f'L{lid}', (x1, max(y1-10,20)),
                    cv2.FONT_HERSHEY_DUPLEX, 0.6, color, 1, cv2.LINE_AA)
        # Save crop
        track_dir = crops_dir / f'track_{lid:03d}'
        track_dir.mkdir(exist_ok=True)
        ts = frame_idx / fps
        fname = f'frame_{frame_idx:06d}_t{ts:.3f}_l{lid}.jpg'
        cv2.imwrite(str(track_dir / fname), det['crop'],
                    [cv2.IMWRITE_JPEG_QUALITY, 95])
        saved_total += 1
        saved_per_track[lid] = saved_per_track.get(lid, 0) + 1
    out.write(annotated)
    pbar.update(1)

pbar.close()
cap.release()
out.release()

elapsed = time.perf_counter() - start
print(f'Done in {elapsed:.1f}s ({elapsed/60:.1f} min)')
print(f'Frames: {frame_idx}, FPS: {frame_idx/elapsed:.2f}')
print(f'Total crops saved: {saved_total}')
print(f'Unique track IDs: {len(saved_per_track)}')

### View the output video

In [ ]:
from IPython.display import Video
Video(config.OUTPUT_VIDEO, embed=True, width=640)